In [1]:
pip install selenium webdriver-manager


In [2]:
pip install webdriver-manager


##Scraping Berita Otomatis dari Liputan6



Kode ini digunakan untuk **mengambil judul, isi, dan kategori berita dari situs Liputan6** secara otomatis. Data yang diperoleh disimpan dalam bentuk **CSV** agar mudah diolah kembali, sekaligus ditampilkan sebagian di layar untuk memastikan hasil scraping berhasil.


In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrap_liputan6(pages=1, output_file="liputan6_news.csv"):
    base_url = "https://www.liputan6.com"
    records = []
    news_id = 1

    for page in range(1, pages+1):
        url = f"{base_url}/news?page={page}"
        r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "html.parser")

        # ambil daftar berita di halaman list
        items = soup.select("article a.articles--iridescent-list--text-item__title-link")
        for a in items:
            link = a["href"]

            # ambil detail berita
            r2 = requests.get(link, headers={"User-Agent":"Mozilla/5.0"})
            soup2 = BeautifulSoup(r2.text, "html.parser")

            judul = soup2.select_one("h1.read-page--header--title").get_text(strip=True) if soup2.select_one("h1.read-page--header--title") else None
            isi = " ".join(p.get_text(strip=True) for p in soup2.select("div.read-page--content-body p"))
            kategori = soup2.select_one("ul.read-page--breadcrumb li:last-child a").get_text(strip=True) if soup2.select_one("ul.read-page--breadcrumb li:last-child a") else None

            records.append({
                "id": news_id,
                "Judul Berita": judul,
                "Isi Berita": isi,
                "Kategori Berita": kategori
            })
            news_id += 1

    # simpan ke CSV
    df = pd.DataFrame(records)
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"✅ Scraping selesai! Data tersimpan di {output_file}")
    return df


# 🚀 Jalankan dan tampilkan hasil
df = scrap_liputan6(pages=1)   # ganti pages=2 atau lebih kalau mau banyak
print(df.head())               # tampilkan 5 berita pertama

# 👉 kalau di Google Colab / Jupyter, bisa download CSV dengan:
# from google.colab import files
# files.download("liputan6_news.csv")


✅ Scraping selesai! Data tersimpan di liputan6_news.csv
   id                                       Judul Berita Isi Berita  \
0   1  Mobile Training Unit Permudah Warga Dapat Kerj...              
1   2  Immanuel Ebenezer Mengaku Terima Setoran Lain ...              
2   3  Top 3 News: Ini Dugaan Tindak Pidana Ferry Irw...              
3   4  KPK Blak-blakan Ungkap Awal Mula Dugaan Ridwan...              
4   5  Menimipas Serahkan Langsung SK Kenaikan Grade ...              

  Kategori Berita  
0            News  
1       Peristiwa  
2       Peristiwa  
3       Peristiwa  
4            News  
